# Resume an agent after its worker stops

Save a receipt, stop the worker mid-task, then restart it and retrieve the same run. Temporal tracks the unfinished work; a worker runs the agent.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/06_durable.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai,temporal] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a5/liteagents-0.3.0a5-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Start a local Temporal service

This starts a test service inside Colab and downloads its binary if needed. It needs no account. The service and files disappear when this runtime is reset.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="liteagents-"))

In [ ]:
import shutil

from temporalio.testing import WorkflowEnvironment

if globals().get("temporal_environment") is not None:
    raise RuntimeError("Run the cleanup cell before starting a new Temporal service.")
temporal_environment = await WorkflowEnvironment.start_local(
    ui=False, dev_server_existing_path=shutil.which("temporal"),
)
temporal_address = temporal_environment.client.service_client.config.target_host
print("Temporal is ready:", temporal_address)

## 4. Create the demo worker

The worker has two tools: save a receipt, then perform a slow check. It runs in a separate process so we can stop it. Expand the cell to inspect the tool definitions and `LiteAgentWorker` call.

In [ ]:
# @title Create the demo worker
import asyncio
import subprocess
import sys

worker_script = workspace / "worker.py"
worker_script.write_text(r'''
import asyncio
import os
import sys
from pathlib import Path
from liteagents import ProfileOptions, Tool, operation_id
from liteagents.temporal import LiteAgentWorker

class SaveReceipt(Tool):
    name = "save_receipt"
    description = "Save a receipt and return its verification code."
    input_schema = {"type": "object", "properties": {}}

    async def execute(self, input):
        receipts = Path("receipts")
        receipts.mkdir(exist_ok=True)
        try:
            with (receipts / operation_id()).open("x") as file:
                file.write("receipt-verified")
            print("Receipt saved once", flush=True)
        except FileExistsError:
            print("Reused receipt", flush=True)
        return "Receipt saved: receipt-verified"

class SlowCheck(Tool):
    name = "slow_check"
    description = "Check the receipt and report its verification result."
    input_schema = {"type": "object", "properties": {}}

    async def execute(self, input):
        print("Slow check started", flush=True)
        await asyncio.sleep(int(sys.argv[1]))
        return "Receipt verification passed: receipt-verified"

profile = ProfileOptions.model_validate_json(os.environ["DEMO_WORKER_PROFILE"])
asyncio.run(LiteAgentWorker(profile=profile, tools=[SaveReceipt(), SlowCheck()], cwd=".").run())
''')
print("Demo worker created.")

## 5. Add Temporal to the profile and start the worker

Clients and workers use the same profile and checkpoint location. The worker gets your provider key from this runtime’s environment.

In [ ]:
from liteagents import LiteAgentClient, LiteAgentOptions, ProfileOptions, TemporalOptions

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
profile.temporal = TemporalOptions(
    address=temporal_address,
    checkpoint_path=str(workspace / "checkpoints.sqlite"),
    heartbeat_timeout_seconds=3,
    activity_timeout_seconds=180,
)
DELAY_SECONDS = 20
worker_command = [sys.executable, "-u", str(worker_script), str(DELAY_SECONDS)]
worker_env = {**os.environ, "DEMO_WORKER_PROFILE": profile.model_dump_json()}
worker_log = workspace / "worker.log"
previous = globals().get("worker")
if previous is not None and previous.poll() is None:
    raise RuntimeError("Run the cleanup cell before starting another worker.")
with worker_log.open("a") as log:
    worker = subprocess.Popen(
        worker_command, cwd=workspace, env=worker_env, stdout=log, stderr=subprocess.STDOUT,
    )
options = LiteAgentOptions(profile=profile, cwd=workspace)
print("Worker started.")

## 6. Submit a job and disconnect

The client closes at the end of this cell. The worker continues the job, and its run ID lets you reconnect.

In [ ]:
async with LiteAgentClient(options=options) as agent:
    handle = await agent.start_run("Call save_receipt once, then slow_check once. Report their results.")
    run_id = handle.run_id
print("Submitted:", run_id)

## 7. Stop the worker, then restart it

This waits for the receipt to be saved and the slow check to start, then kills only our demo worker. The replacement uses the same checkpoints.

In [ ]:
async with asyncio.timeout(120):
    while "Slow check started" not in worker_log.read_text():
        if worker.poll() is not None:
            raise RuntimeError("Worker exited. Read worker_log to see why.")
        await asyncio.sleep(0.2)
worker.kill()
worker.wait(timeout=10)
print("Stopped the worker during the check.")

with worker_log.open("a") as log:
    worker = subprocess.Popen(
        worker_command, cwd=workspace, env=worker_env, stdout=log, stderr=subprocess.STDOUT,
    )
print("Restarted the worker.")

## 8. Retrieve the original job

`get_run()` attaches without submitting again. The completed receipt save is reused; the interrupted check runs again.

In [ ]:
try:
    async with asyncio.timeout(240):
        async with LiteAgentClient(options=options) as agent:
            handle = await agent.get_run(run_id)
            result = await handle.result()
    print(result.text)
finally:
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=10)
        except subprocess.TimeoutExpired:
            worker.kill()
            worker.wait()

print("Receipts saved:", len(list((workspace / "receipts").glob("*"))))
print("Check attempts:", worker_log.read_text().count("Slow check started"))

## 9. Clean up

Run this even if you interrupt an earlier step. It stops only this notebook’s worker and test service.

In [ ]:
process = globals().get("worker")
if process is not None and process.poll() is None:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
if temporal_environment is not None:
    await temporal_environment.shutdown()
    temporal_environment = None

Expected result: **1 receipt, 2 check attempts**, and the final answer. The receipt tool also uses `operation_id()` to guard against a duplicate external write.

This demonstrates recovery from worker loss while the service and files survive. A permanent deployment needs persistent storage and a worker outside Colab; see [self-hosting](https://github.com/BerriAI/liteagents/blob/main/docs/self-hosting.md).

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)